In [6]:
import json
import re
from collections.abc import Callable
from pathlib import Path

TICKERS = ("C", "JPM", "WFC")
SHORT_TABLE_CHARS = 120

BULLET_PATTERN = re.compile(
    r"(^|\n)\s*(?:[\u2022\u25aa\u25e6\u2023-])\s+"
)


def find_project_root() -> Path:
    current_path = Path.cwd().resolve()

    for candidate in (current_path, *current_path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Nije pronađen pyproject.toml. Sačuvaj notebook unutar projekta."
    )


def load_tables(project_root: Path, ticker: str) -> list[dict]:
    path = (
        project_root
        / "data"
        / "processed"
        / "elements"
        / f"{ticker.lower()}.jsonl"
    )

    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    return [
        add_table_metrics(record)
        for record in records
        if record["element_type"] == "table"
    ]


def add_table_metrics(record: dict) -> dict:
    text = str(record["text"]).strip()
    rows = [row.strip() for row in text.splitlines() if row.strip()]
    plain_text = " ".join(text.split())

    content_columns = max(
        (
            sum(bool(cell.strip()) for cell in row.split(" | "))
            for row in rows
        ),
        default=0,
    )

    word_count = len(re.findall(r"\b[\w'-]+\b", plain_text))
    has_numeric_value = bool(re.search(r"\d", plain_text))
    has_sentence_end = bool(re.search(r"[.!?](?:\s|$)", plain_text))

    return {
        **record,
        "row_count": len(rows),
        "content_columns": content_columns,
        "char_count": len(plain_text),
        "word_count": word_count,
        "has_numeric_value": has_numeric_value,
        "bullet_like": bool(BULLET_PATTERN.search(text)),
        "title_like": (
            len(rows) <= 2
            and content_columns <= 1
            and len(plain_text) <= SHORT_TABLE_CHARS
            and word_count <= 15
            and not has_numeric_value
            and not plain_text.endswith((".", "!", "?", ";"))
        ),
        "paragraph_like": (
            content_columns <= 1
            and len(plain_text) > SHORT_TABLE_CHARS
            and has_sentence_end
        ),
    }


def shorten(text: str, limit: int = 220) -> str:
    text = " ".join(text.split())
    return text if len(text) <= limit else f"{text[:limit].rstrip()}..."


project_root = find_project_root()
tables_by_ticker = {
    ticker: load_tables(project_root, ticker)
    for ticker in TICKERS
}

categories: dict[str, Callable[[dict], bool]] = {
    "one_row": lambda table: table["row_count"] == 1,
    "one_content_column": lambda table: table["content_columns"] <= 1,
    "very_short": lambda table: table["char_count"] <= SHORT_TABLE_CHARS,
    "no_numeric_values": lambda table: not table["has_numeric_value"],
    "title_like": lambda table: table["title_like"],
    "bullet_like": lambda table: table["bullet_like"],
    "paragraph_like": lambda table: table["paragraph_like"],
}

print(f"Project root: {project_root}")
print("\nCOUNTS")

for ticker, tables in tables_by_ticker.items():
    print(f"\n{ticker}: tables={len(tables)}")

    for category, matches in categories.items():
        count = sum(matches(table) for table in tables)
        print(f"  {category:<20} {count:>5}")


def select_examples(
    category: str,
    limit: int = 8,
) -> list[tuple[str, dict]]:
    matches = categories[category]
    pools = {
        ticker: sorted(
            (table for table in tables if matches(table)),
            key=lambda table: table["char_count"],
            reverse=True,
        )
        for ticker, tables in tables_by_ticker.items()
    }

    examples: list[tuple[str, dict]] = []

    while len(examples) < limit:
        added = False

        for ticker in TICKERS:
            if pools[ticker] and len(examples) < limit:
                examples.append((ticker, pools[ticker].pop(0)))
                added = True

        if not added:
            break

    return examples


print("\nEXAMPLES")

example_categories = (
    "one_row",
    "one_content_column",
    "title_like",
    "bullet_like",
    "paragraph_like",
)

for category in example_categories:
    print(f"\n[{category}]")

    examples = select_examples(category)

    if not examples:
        print("  Nema primera.")
        continue

    for ticker, table in examples:
        flags = [
            name
            for name, matches in categories.items()
            if matches(table)
        ]

        print(
            f"  {ticker} order={table['order_index']} "
            f"rows={table['row_count']} "
            f"cols={table['content_columns']} "
            f"chars={table['char_count']} "
            f"flags={','.join(flags)}"
        )
        print(f"    {shorten(str(table['text']))}")

Project root: C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\Repositories\Banking-Technology-and-Operational-Risk-Intelligence-Assistant

COUNTS

C: tables=319
  one_row                  7
  one_content_column       0
  very_short              11
  no_numeric_values        6
  title_like               0
  bullet_like              0
  paragraph_like           0

JPM: tables=638
  one_row                331
  one_content_column       1
  very_short             334
  no_numeric_values        3
  title_like               0
  bullet_like              2
  paragraph_like           0

WFC: tables=16
  one_row                  0
  one_content_column       0
  very_short               2
  no_numeric_values        2
  title_like               0
  bullet_like              0
  paragraph_like           0

EXAMPLES

[one_row]
  C order=3921 rows=1 cols=3 chars=985 flags=one_row
    Titi Cole Former Head of Legacy Franchises, Citigroup Inc. Ellen M. Costello Chair, Citibank, N.A. Grace E. Dai

In [7]:
import importlib

import bankscope.parsing.sec_html_parser as sec_html_parser

importlib.reload(sec_html_parser)
normalize_text = sec_html_parser.normalize_text

In [8]:

synthetic_cases = {
    "\ufeffOperational risk": "Operational risk",
    "cyber\u200bsecurity\u2060 risk": "cybersecurity risk",
    "inter\u00adnational operations": "international operations",
    "non\u00a0breaking\u202fspaces": "non breaking spaces",
}

for original, expected in synthetic_cases.items():
    result = normalize_text(original)
    assert result == expected, (repr(original), repr(result), repr(expected))

preserved_cases = (
    "U.S. GAAP legal‑entity net income",
    "Yes ¨ No þ",
    "Risk exposure: $1,234.50",
)

for original in preserved_cases:
    result = normalize_text(original)
    assert result == original, (repr(original), repr(result))

print("Synthetic Unicode checks passed.")

print("Synthetic Unicode checks passed.")

for ticker in ("c", "jpm", "wfc"):
    path = project_root / "data" / "processed" / "elements" / f"{ticker}.jsonl"
    changes = []

    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        record = json.loads(line)
        text = str(record["text"])

        old_normalized = " ".join(text.split())
        new_normalized = normalize_text(text)

        if old_normalized != new_normalized:
            changes.append(
                (
                    record["order_index"],
                    old_normalized,
                    new_normalized,
                )
            )

    print(f"\n{ticker.upper()}: unicode_changes={len(changes)}")

    for order_index, before, after in changes[:5]:
        print(f"  order={order_index}")
        print(f"    before: {before[:180]!r}")
        print(f"    after:  {after[:180]!r}")

Synthetic Unicode checks passed.
Synthetic Unicode checks passed.

C: unicode_changes=0

JPM: unicode_changes=0

WFC: unicode_changes=0


In [9]:
import importlib
import json
from collections import Counter
from pathlib import Path

import bankscope.parsing.sec_html_parser as sec_html_parser

importlib.reload(sec_html_parser)

for ticker in ("c", "jpm", "wfc"):
    path = project_root / "data" / "processed" / "elements" / f"{ticker}.jsonl"

    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    converted = []

    for record in records:
        if record["element_type"] != "table":
            continue

        new_type = sec_html_parser.classify_table_element(record["text"])

        if new_type != "table":
            converted.append(
                {
                    "order_index": record["order_index"],
                    "new_type": new_type,
                    "text": record["text"],
                }
            )

    counts = Counter(record["new_type"] for record in converted)

    print(f"\n{ticker.upper()}: converted={len(converted)} {dict(counts)}")

    for new_type in ("heading", "list", "paragraph"):
        examples = [
            record for record in converted
            if record["new_type"] == new_type
        ][:3]

        for record in examples:
            text = " ".join(record["text"].split())
            print(
                f"  {new_type:<9} order={record['order_index']} "
                f"{text[:180]!r}"
            )


C: converted=0 {}

JPM: converted=0 {}

WFC: converted=0 {}


In [10]:
importlib.reload(sec_html_parser)

manifest_path = project_root / "artifacts" / "manifests" / "filings.json"
filings = json.loads(manifest_path.read_text(encoding="utf-8"))

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])
    if not html_path.is_absolute():
        html_path = project_root / html_path
    elements = sec_html_parser.parse_filing_html(html_path)
    navigation = [
        element
        for element in elements
        if element["is_navigation"]
    ]

    type_counts = {}

    for element in navigation:
        element_type = str(element["element_type"])
        type_counts[element_type] = type_counts.get(element_type, 0) + 1

    print(
        f"\n{ticker}: navigation={len(navigation)} "
        f"types={type_counts}"
    )

    for element in navigation[:10]:
        text = " ".join(str(element["text"]).split())

        print(
            f"  order={element['order_index']} "
            f"type={element['element_type']} "
            f"{text[:220]!r}"
        )


JPM: navigation=2 types={'table': 1, 'paragraph': 1}
  order=29 type=table 'Part I | | Page Item 1. | Business . | 1 | Overview | 1 | Business segments & Corporate | 1 | Competition | 1 | Supervision and regulation | 2-6 | Human capital | 7-8 | Distribution of assets, liabilities and stockholder'
  order=909 type=paragraph 'Table of contents'

C: navigation=1 types={'paragraph': 1}
  order=33 type=paragraph 'FORM 10-K CROSS-REFERENCE INDEX'

WFC: navigation=0 types={}


In [11]:
from bs4 import BeautifulSoup
from bs4.element import Tag

HEADING_TAGS = ("h1", "h2", "h3", "h4", "h5", "h6")
CENTER_PATTERN = re.compile(r"text-align\s*:\s*center", re.IGNORECASE)


def heading_signals(tag: Tag, text: str) -> list[str]:
    signals = []

    if tag.name in HEADING_TAGS:
        signals.append("heading_tag")

    style = str(tag.get("style", ""))
    align = str(tag.get("align", "")).casefold()

    if align == "center" or CENTER_PATTERN.search(style):
        signals.append("centered")

    bold_tag = tag.find(["b", "strong"])

    if (
        bold_tag is not None
        and sec_html_parser.normalize_text(
            bold_tag.get_text(" ", strip=True)
        )
        == text
    ):
        signals.append("fully_bold")

    letters = [character for character in text if character.isalpha()]

    if (
        len(letters) >= 4
        and all(character.isupper() for character in letters)
    ):
        signals.append("uppercase")

    return signals


for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    soup = BeautifulSoup(html_path.read_bytes(), "lxml")

    for tag in soup(["script", "style", "noscript", "ix:header"]):
        tag.decompose()

    for tag in soup.find_all(style=sec_html_parser.HIDDEN_STYLE_PATTERN):
        tag.decompose()

    root = soup.body or soup
    candidates = []

    for tag in root.find_all(
        ("div", "p", "h1", "h2", "h3", "h4", "h5", "h6")
    ):
        if tag.find_parent("table") is not None:
            continue

        text = sec_html_parser.normalize_text(
            tag.get_text(" ", strip=True)
        )

        if (
            not text
            or len(text) > 180
            or len(sec_html_parser.WORD_PATTERN.findall(text)) > 24
            or text.endswith((".", "!", "?", ";"))
        ):
            continue

        signals = heading_signals(tag, text)

        if signals:
            candidates.append((tag.name, signals, text))

    signal_counts = {}

    for _, signals, _ in candidates:
        for signal in signals:
            signal_counts[signal] = signal_counts.get(signal, 0) + 1

    print(
        f"\n{ticker}: candidates={len(candidates)} "
        f"signals={signal_counts}"
    )

    for tag_name, signals, text in candidates[:20]:
        print(
            f"  tag={tag_name:<3} "
            f"signals={','.join(signals):<30} "
            f"{text[:180]!r}"
        )

C:\Users\nikola.bakic\AppData\Local\Temp\ipykernel_7460\1008938807.py:53: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_path.read_bytes(), "lxml")



JPM: candidates=348 signals={'centered': 316, 'uppercase': 37}
  tag=div signals=centered,uppercase             'UNITED STATES'
  tag=div signals=centered,uppercase             'SECURITIES AND EXCHANGE COMMISSION'
  tag=div signals=centered,uppercase             'WASHINGTON, D.C. 20549'
  tag=div signals=centered,uppercase             'FORM 10-K'
  tag=div signals=centered                       'Annual report pursuant to Section 13 or 15(d) of'
  tag=div signals=centered                       'the Securities Exchange Act of 1934'
  tag=div signals=centered                       'For the fiscal year ended Commission file December 31 , 2025 number 1-5805'
  tag=div signals=centered                       '(Exact name of registrant as specified in its charter)'
  tag=div signals=centered                       'Registrant’s telephone number, including area code: ( 212 ) 270-6000'
  tag=div signals=centered                       'Securities registered pursuant to Section 12(b) of the Act:'


In [12]:
importlib.reload(sec_html_parser)

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    elements = sec_html_parser.parse_filing_html(html_path)

    headings = [
        element
        for element in elements
        if element["element_type"] == "heading"
        and not element["is_navigation"]
    ]

    section_titles = {
        str(element["section_title"])
        for element in elements
        if element["section_title"] is not None
    }

    print(
        f"\n{ticker}: elements={len(elements)} "
        f"headings={len(headings)} "
        f"section_titles={len(section_titles)}"
    )

    step = max(1, len(headings) // 10)

    for element in headings[::step][:10]:
        text = " ".join(str(element["text"]).split())

        print(
            f"  order={element['order_index']} "
            f"sec_item={element['sec_item']!r} "
            f"title={text[:180]!r}"
        )


JPM: elements=4593 headings=60 section_titles=60
  order=0 sec_item=None title='UNITED STATES'
  order=803 sec_item='Item 1B' title='Item 1B. Unresolved Staff Comments.'
  order=831 sec_item='Item 6' title='Item 6. Reserved'
  order=849 sec_item='Item 9B' title='Item 9B. Other Information.'
  order=888 sec_item='Item 14' title='Item 14. Principal Accounting Fees and Services.'
  order=1087 sec_item='Item 15' title='CONSOLIDATED BALANCE SHEETS AND CASH FLOWS ANALYSIS'
  order=1375 sec_item='Item 15' title='CORPORATE'
  order=1831 sec_item='Item 15' title='REPUTATION RISK MANAGEMENT'
  order=2092 sec_item='Item 15' title='INVESTMENT PORTFOLIO RISK MANAGEMENT'
  order=2371 sec_item='Item 15' title='CONDUCT RISK MANAGEMENT'

C: elements=4159 headings=123 section_titles=120
  order=0 sec_item=None title='UNITED STATES'
  order=162 sec_item=None title='SUMMARY OF SELECTED FINANCIAL DATA'
  order=624 sec_item=None title='RISK FACTORS'
  order=967 sec_item=None title='CORPORATE CREDIT'
  orde

In [13]:
importlib.reload(sec_html_parser)

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    elements = [
        element
        for element in sec_html_parser.parse_filing_html(html_path)
        if not element["is_navigation"]
    ]

    sec_item_sequence = []

    for element in elements:
        sec_item = element["sec_item"]

        if (
            sec_item is not None
            and (
                not sec_item_sequence
                or sec_item != sec_item_sequence[-1]
            )
        ):
            sec_item_sequence.append(sec_item)

    heading_indices = [
        index
        for index, element in enumerate(elements)
        if element["element_type"] == "heading"
    ]

    heading_spans = []

    for position, start_index in enumerate(heading_indices):
        end_index = (
            heading_indices[position + 1]
            if position + 1 < len(heading_indices)
            else len(elements)
        )

        content = elements[start_index + 1 : end_index]
        content_word_count = sum(
            len(
                sec_html_parser.WORD_PATTERN.findall(
                    str(element["text"])
                )
            )
            for element in content
        )

        heading_spans.append(
            {
                "heading": str(elements[start_index]["text"]),
                "content_elements": len(content),
                "content_words": content_word_count,
            }
        )

    short_spans = [
        span
        for span in heading_spans
        if span["content_elements"] <= 2
        or span["content_words"] < 40
    ]

    print(
        f"\n{ticker}: "
        f"sec_item_transitions={len(sec_item_sequence)} "
        f"unique_items={len(set(sec_item_sequence))} "
        f"heading_spans={len(heading_spans)} "
        f"short_spans={len(short_spans)}"
    )
    print(f"  sec_items={sec_item_sequence}")

    for span in short_spans[:12]:
        print(
            f"  elements={span['content_elements']:<3} "
            f"words={span['content_words']:<4} "
            f"title={span['heading'][:160]!r}"
        )


JPM: sec_item_transitions=22 unique_items=22 heading_spans=60 short_spans=17
  sec_items=['Item 1', 'Item 1A', 'Item 1B', 'Item 1C', 'Item 2', 'Item 3', 'Item 4', 'Item 5', 'Item 6', 'Item 7', 'Item 7A', 'Item 8', 'Item 9', 'Item 9A', 'Item 9B', 'Item 9C', 'Item 10', 'Item 11', 'Item 12', 'Item 13', 'Item 14', 'Item 15']
  elements=0   words=0    title='UNITED STATES'
  elements=0   words=0    title='SECURITIES AND EXCHANGE COMMISSION'
  elements=0   words=0    title='WASHINGTON, D.C. 20549'
  elements=1   words=1    title='Item 1B. Unresolved Staff Comments.'
  elements=1   words=23   title='Item 1C. Cybersecurity.'
  elements=1   words=14   title='Item 3. Legal Proceedings.'
  elements=2   words=3    title='Item 4. Mine Safety Disclosures.'
  elements=0   words=0    title='Item 6. Reserved'
  elements=1   words=44   title='Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations.'
  elements=3   words=32   title='Item 7A. Quantitative and Qualita

In [14]:
importlib.reload(sec_html_parser)

unit_pattern = re.compile(
    r"\b(?:dollars?|amounts?)\s+in\s+"
    r"(?:thousands?|millions?|billions?)\b"
    r"|\bin\s+(?:thousands?|millions?|billions?)\b",
    re.IGNORECASE,
)

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    elements = sec_html_parser.parse_filing_html(html_path)

    tables = [
        (index, element)
        for index, element in enumerate(elements)
        if element["element_type"] == "table"
        and not element["is_navigation"]
    ]

    with_section_title = sum(
        table["section_title"] is not None
        for _, table in tables
    )

    with_nearby_unit = 0

    for index, _ in tables:
        nearby_text = " ".join(
            str(element["text"])
            for element in elements[max(0, index - 3) : index]
            if not element["is_navigation"]
        )

        if unit_pattern.search(nearby_text):
            with_nearby_unit += 1

    print(
        f"\n{ticker}: tables={len(tables)} "
        f"with_section_title={with_section_title} "
        f"with_nearby_unit={with_nearby_unit}"
    )

    if not tables:
        continue

    sample_positions = sorted(
        {
            0,
            len(tables) // 4,
            len(tables) // 2,
            3 * len(tables) // 4,
            len(tables) - 1,
        }
    )

    for position in sample_positions:
        index, table = tables[position]

        previous_elements = [
            element
            for element in elements[max(0, index - 3) : index]
            if not element["is_navigation"]
        ]

        print(
            f"\n  table_order={table['order_index']} "
            f"section_title={table['section_title']!r}"
        )

        for previous in previous_elements:
            text = " ".join(str(previous["text"]).split())

            print(
                f"    previous[{previous['element_type']}]: "
                f"{text[:180]!r}"
            )

        table_rows = [
            row.strip()
            for row in str(table["text"]).splitlines()
            if row.strip()
        ]

        for row in table_rows[:3]:
            print(f"    row: {row[:220]!r}")


JPM: tables=637 with_section_title=637 with_nearby_unit=143

  table_order=6 section_title='FORM 10-K'
    previous[heading]: 'FORM 10-K'
    previous[paragraph]: 'Annual report pursuant to Section 13 or 15(d) of'
    previous[paragraph]: 'the Securities Exchange Act of 1934'
    row: 'For the fiscal year ended |  | Commission file |'
    row: 'December 31 , 2025 |  | number | 1-5805 |'

  table_order=1608 section_title='CAPITAL RISK MANAGEMENT'
    previous[paragraph]: '(f) As of December 31, 2025 and 2024, included an incremental $468 million and $541 million allowance for credit losses, respectively, on certain assets associated with First Repub'
    previous[paragraph]: 'Capital rollforward'
    previous[paragraph]: 'The following table presents the changes in CET1 capital, Tier 1 capital and Tier 2 capital for the year ended December 31, 2025.'
    row: 'Year ended December 31, (in millions) | 2025'
    row: 'Standardized/Advanced CET1 capital at December 31, 2024 | $ | 275,513 |

In [15]:
import json
from pathlib import Path

import sec2md

from bankscope.parsing.sec_html_parser import parse_filing_html

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

manifest_path = project_root / "artifacts/manifests/filings.json"
filings = json.loads(manifest_path.read_text(encoding="utf-8"))

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    current_elements = parse_filing_html(html_path)

    pages = sec2md.parse_filing(
        html_path.read_bytes(),
        include_elements=True,
    )

    sec2md_elements = [
        element
        for page in pages
        for element in (page.elements or [])
    ]

    current_tables = [
        element
        for element in current_elements
        if element["element_type"] == "table"
    ]

    sec2md_tables = [
        element
        for element in sec2md_elements
        if element.kind == "table"
    ]

    print(
        f"\n{ticker}: "
        f"current_elements={len(current_elements)} "
        f"current_tables={len(current_tables)} "
        f"sec2md_pages={len(pages)} "
        f"sec2md_elements={len(sec2md_elements)} "
        f"sec2md_tables={len(sec2md_tables)}"
    )

    if not sec2md_tables:
        continue

    sample_positions = sorted(
        {
            0,
            len(sec2md_tables) // 2,
            len(sec2md_tables) - 1,
        }
    )

    for position in sample_positions:
        table = sec2md_tables[position]

        print(
            f"\n  page={table.page_start} "
            f"tokens={table.tokens}"
        )

        for line in table.content.splitlines()[:6]:
            print(f"    {line[:220]}")

c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")



JPM: current_elements=4593 current_tables=638 sec2md_pages=330 sec2md_elements=1767 sec2md_tables=350

  page=1 tokens=67
    **FORM 10-K**
    
    **Annual report pursuant to Section 13 or 15(d) of**
    
    **the Securities Exchange Act of 1934**
    

  page=185 tokens=1352
    | Level 3 inputs (a) |  |  |  |  |  |  |
    | --- | --- | --- | --- | --- | --- | --- |
    | December 31, 2025 |  |  |  |  |  |  |
    | Product/Instrument | Fair value (in millions) | Principal valuation technique | Unobservable inputs (g) | Range of input values | Average (i) |  |
    | Residential mortgage-backed securities and loans (b) | $ 889 | Discounted cash flows | Yield | 0 % | 70 % | 7 % |
    |  |  |  | Prepayment speed | 7 % | 14 % | 9 % |

  page=330 tokens=490
    Pursuant to the requirements of the Securities Exchange Act of 1934, this report has been signed below by the following persons on behalf of the registrant and in the capacity and on the date indicated. JPMorgan Chase & 
    
   

c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")



C: current_elements=4159 current_tables=319 sec2md_pages=318 sec2md_elements=1706 sec2md_tables=326

  page=1 tokens=41
    **FORM 10-K**
    
    **(Mark One)**
    
    ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

  page=189 tokens=242
    **Income Taxes Paid**
    
    Details of the Company’s income taxes paid are presented below:
    
    | In millions of dollars | 2025 | 2024 | 2023 |
    | --- | --- | --- | --- |

  page=312 tokens=128
    **/s/ Nicole Giles**
    
    Nicole Giles
    
    The Directors of Citigroup listed below executed a power of attorney appointing Mark A. L. Mason their attorney-in-fact, empowering him to sign this report on their behalf.
    

WFC: current_elements=289 current_tables=16 sec2md_pages=21 sec2md_elements=128 sec2md_tables=40

  page=1 tokens=400
    **No. 41-0449260**
    
    (I.R.S. Employer Identification No.)
    
    **333 Market Street , San Francisco , California 94105**
    

  page=11 token

c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")


In [16]:
import re

number_pattern = re.compile(r"\d[\d,.$%()-]*")

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    current_elements = parse_filing_html(html_path)
    pages = sec2md.parse_filing(
        html_path.read_bytes(),
        include_elements=True,
    )

    sec2md_elements = [
        element
        for page in pages
        for element in (page.elements or [])
    ]

    current_text = "\n".join(
        str(element["text"])
        for element in current_elements
        if not element["is_navigation"]
    )
    sec2md_text = "\n".join(
        element.content
        for element in sec2md_elements
    )

    current_numbers = number_pattern.findall(current_text)
    sec2md_numbers = number_pattern.findall(sec2md_text)

    print(
        f"{ticker}: "
        f"current_chars={len(current_text):,} "
        f"sec2md_chars={len(sec2md_text):,} "
        f"current_numbers={len(current_numbers):,} "
        f"sec2md_numbers={len(sec2md_numbers):,} "
        f"char_ratio={len(sec2md_text) / max(len(current_text), 1):.2f} "
        f"number_ratio={len(sec2md_numbers) / max(len(current_numbers), 1):.2f}"
    )

c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")


JPM: current_chars=1,279,267 sec2md_chars=1,243,375 current_numbers=18,217 sec2md_numbers=17,419 char_ratio=0.97 number_ratio=0.96


c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")


C: current_chars=1,258,003 sec2md_chars=1,239,216 current_numbers=21,937 sec2md_numbers=21,671 char_ratio=0.99 number_ratio=0.99
WFC: current_chars=88,138 sec2md_chars=91,009 current_numbers=953 sec2md_numbers=956 char_ratio=1.03 number_ratio=1.00


c:\venvs\bankscope\Lib\site-packages\sec2md\parser.py:40: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  self.soup = BeautifulSoup(content, "lxml")


In [17]:
import json
from pathlib import Path

chunk_path = Path("C:\\Users\\nikola.bakic\\OneDrive - Sixsentix AG\\Documents\\Repositories\\Banking-Technology-and-Operational-Risk-Intelligence-Assistant\\data\\processed\\chunks\\sec_10k_chunks.jsonl")

table_chunks = [
    json.loads(line)
    for line in chunk_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and json.loads(line).get("element_type") == "table"
]

print(f"Table chunks: {len(table_chunks)}")
print("Available fields:")
print(sorted(table_chunks[0]))

selected_chunks = []
seen_tickers = set()

for chunk in table_chunks:
    ticker = chunk.get("ticker")

    if ticker not in seen_tickers:
        selected_chunks.append(chunk)
        seen_tickers.add(ticker)

    if len(selected_chunks) == 5:
        break

for chunk in selected_chunks:
    print("\n" + "=" * 80)
    for field in (
        "chunk_id",
        "ticker",
        "report_date",
        "sec_item",
        "section_title",
        "table_id",
        "table_part_index",
        "table_part_count",
        "table_header",
        "table_context",
    ):
        print(f"{field}: {chunk.get(field)}")

    print(f"text preview: {chunk.get('text', '')[:800]}")

Table chunks: 3220
Available fields:
['accession_number', 'chunk_id', 'chunk_index', 'cik', 'element_type', 'filing_date', 'order_end', 'order_start', 'report_date', 'sec_item', 'section_title', 'source_url', 'table_context', 'table_header', 'table_id', 'table_part_count', 'table_part_index', 'text', 'ticker', 'token_count']

chunk_id: b17353bab238df4544a5654f3c656fc9f85af190af3da0b14b79d40aeed8d99c
ticker: ALLY
report_date: 2025-12-31
sec_item: None
section_title: SECURITIES AND EXCHANGE COMMISSION
table_id: 9c12b1e362d861f8aac090c258bd8eb03eb9d60b3f7027368354ac80b6d3072b
table_part_index: 0
table_part_count: 1
table_header: ☑ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
table_context: Section: SECURITIES AND EXCHANGE COMMISSION
text preview: ☑ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

chunk_id: 8a968674cc28b6e9bbe1a16db04283de8c6443e4c4f73131c788461a401210db
ticker: BAC
report_date: 2025-12-31
sec

In [18]:
import re
from collections.abc import Callable

Chunk = dict[str, object]

selectors: list[tuple[str, Callable[[Chunk], bool]]] = [
    (
        "employee_or_workforce",
        lambda chunk: bool(
            re.search(
                r"\b(employee|employees|workforce|engagement)\b",
                str(chunk.get("text", "")),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "stock_repurchase",
        lambda chunk: bool(
            re.search(
                r"\b(repurchases?|shares? purchased)\b",
                str(chunk.get("text", "")),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "tier_1_capital",
        lambda chunk: bool(
            re.search(
                r"\btier\s+1\b",
                str(chunk.get("text", "")),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "level_3_assets",
        lambda chunk: bool(
            re.search(
                r"\blevel\s+3\b",
                str(chunk.get("text", "")),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "past_due",
        lambda chunk: bool(
            re.search(
                r"\b\d+\s*[-–]\s*\d+\s+days?\s+past\s+due\b",
                str(chunk.get("text", "")),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "basis_points",
        lambda chunk: bool(
            re.search(
                r"\bbasis\s+points?\b",
                " ".join(
                    [
                        str(chunk.get("table_context", "")),
                        str(chunk.get("table_header", "")),
                        str(chunk.get("text", "")),
                    ]
                ),
                flags=re.IGNORECASE,
            )
        ),
    ),
    (
        "split_table",
        lambda chunk: int(chunk.get("table_part_count") or 1) > 1,
    ),
    (
        "multirow_header",
        lambda chunk: len(
            [
                line
                for line in str(chunk.get("table_header", "")).splitlines()
                if line.strip()
            ]
        )
        > 1,
    ),
    (
        "missing_context",
        lambda chunk: not str(chunk.get("table_context") or "").strip(),
    ),
]

selected: list[tuple[str, Chunk]] = []
used_chunk_ids: set[str] = set()

for category, selector in selectors:
    match = next(
        (
            chunk
            for chunk in table_chunks
            if selector(chunk)
            and str(chunk["chunk_id"]) not in used_chunk_ids
        ),
        None,
    )

    if match is not None:
        selected.append((category, match))
        used_chunk_ids.add(str(match["chunk_id"]))

for category, chunk in selected:
    print("\n" + "=" * 100)
    print(f"category: {category}")
    print(f"ticker: {chunk.get('ticker')}")
    print(f"chunk_id: {chunk.get('chunk_id')}")
    print(f"table_id: {chunk.get('table_id')}")
    print(
        "table_part: "
        f"{chunk.get('table_part_index')} / "
        f"{chunk.get('table_part_count')}"
    )
    print(f"sec_item: {chunk.get('sec_item')}")
    print(f"section_title: {chunk.get('section_title')}")
    print(f"table_context: {chunk.get('table_context')}")
    print(f"table_header:\n{chunk.get('table_header')}")
    print(f"text preview:\n{str(chunk.get('text', ''))[:1500]}")


category: employee_or_workforce
ticker: ALLY
chunk_id: fd3f225bb1c1723c73879f138194c84a2dc1523d2b3bc2902616464c1bcb8e41
table_id: 6dbea8f594d6de768999a3795f3a2cd83e2160f7700079f8f14b7b4213d147d4
table_part: 0 / 1
sec_item: Item 1
section_title: Item 1. Business
table_context: Section: Item 1. Business
Description: The following table indicates our company-wide engagement survey results as measured by our third-party provider, based on a 100-point scale, as well as our participation rates for the survey.
table_header:
|  | 2025 |  | 2024
text preview:
|  | 2025 |  | 2024
Ally score |  | 84 |  | 83
Financial services benchmark |  | 77 |  | 76
Ally employee participation % (a) |  | 81 |  | 80

category: stock_repurchase
ticker: ALLY
chunk_id: d2637adc44316d6bf5a99095fe6a6bb431af06c057d3392dc7503fb1ac2f55e5
table_id: df468c7e739105d761b36c4bbfe046b396177aec76c261af9108ae3e6d089433
table_part: 0 / 3
sec_item: Item 8
section_title: Item 8. Financial Statements and Supplementary Data
table_c

In [20]:
import json
from pathlib import Path

sample_path = Path(
    "C:\\Users\\nikola.bakic\\OneDrive - Sixsentix AG\\Documents\\Repositories\\Banking-Technology-and-Operational-Risk-Intelligence-Assistant\\data\\processed\\chunks\\table_proxy_sample.jsonl"
)
sample_path.parent.mkdir(parents=True, exist_ok=True)

sample_chunks = [chunk for _, chunk in selected]

sample_path.write_text(
    "".join(
        json.dumps(chunk, ensure_ascii=False) + "\n"
        for chunk in sample_chunks
    ),
    encoding="utf-8",
)

print(f"Saved sample chunks: {len(sample_chunks)}")

Saved sample chunks: 9


In [21]:
import json
from pathlib import Path

chunk_path = Path("C:\\Users\\nikola.bakic\\OneDrive - Sixsentix AG\\Documents\\Repositories\\Banking-Technology-and-Operational-Risk-Intelligence-Assistant\\data\\processed\\chunks\\sec_10k_chunks.jsonl")
sample_path = Path("C:\\Users\\nikola.bakic\\OneDrive - Sixsentix AG\\Documents\\Repositories\\Banking-Technology-and-Operational-Risk-Intelligence-Assistant\\data\\processed\\chunks\\table_proxy_sample.jsonl")

table_chunks = [
    json.loads(line)
    for line in chunk_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
    and json.loads(line).get("element_type") == "table"
]

sample_chunks = [
    json.loads(line)
    for line in sample_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

used_chunk_ids = {
    str(chunk["chunk_id"])
    for chunk in sample_chunks
}

preferred_tickers = ("BAC", "C", "GS", "JPM", "LOB", "PNC", "TFC")

for ticker in preferred_tickers:
    candidates = [
        chunk
        for chunk in table_chunks
        if chunk.get("ticker") == ticker
        and str(chunk.get("chunk_id")) not in used_chunk_ids
        and int(chunk.get("token_count") or 0) >= 150
        and str(chunk.get("table_header") or "").strip()
        and (
            str(chunk.get("sec_item") or "") in {"Item 7", "Item 8"}
            or "Description:" in str(chunk.get("table_context") or "")
        )
    ]

    if not candidates:
        continue

    candidate = max(
        candidates,
        key=lambda chunk: (
            "Description:" in str(chunk.get("table_context") or ""),
            int(chunk.get("table_part_count") or 1) > 1,
            int(chunk.get("token_count") or 0),
        ),
    )

    sample_chunks.append(candidate)
    used_chunk_ids.add(str(candidate["chunk_id"]))

    if len({chunk.get("ticker") for chunk in sample_chunks}) >= 6:
        break

sample_path.write_text(
    "".join(
        json.dumps(chunk, ensure_ascii=False) + "\n"
        for chunk in sample_chunks
    ),
    encoding="utf-8",
)

print(f"Sample chunks: {len(sample_chunks)}")
print(f"Tickers: {sorted({chunk.get('ticker') for chunk in sample_chunks})}")
print(f"Saved to: {sample_path.resolve()}")

Sample chunks: 14
Tickers: ['ALLY', 'BAC', 'C', 'GS', 'JPM', 'LOB']
Saved to: C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\Repositories\Banking-Technology-and-Operational-Risk-Intelligence-Assistant\data\processed\chunks\table_proxy_sample.jsonl
